# Analyzing Historical Stock/Revenue Data and Building a Dashboard
**IBM Python Project for Data Science – Final Assignment**

This notebook extracts Tesla and GameStop stock data with `yfinance`, scrapes their quarterly revenue with `BeautifulSoup`, and builds a dashboard for each company with Plotly.

**How to run:** use *Runtime → Run all* (Colab) or *Kernel → Restart & Run All* (Jupyter). Run the cells in order from the top.

## Setup: Install and Import Libraries

In [ ]:
!pip install yfinance pandas requests beautifulsoup4 plotly lxml nbformat --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Setup: Define the `make_graph` Function

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    # Work on copies and normalise date columns (yfinance dates can be timezone-aware)
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()
    stock_data["Date"] = pd.to_datetime(stock_data["Date"]).dt.tz_localize(None)
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"])

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=.3
    )

    stock_data_specific = stock_data[stock_data["Date"] <= pd.to_datetime("2021-06-14")]
    revenue_data_specific = revenue_data[revenue_data["Date"] <= pd.to_datetime("2021-04-30")]

    fig.add_trace(go.Scatter(x=stock_data_specific["Date"],
                             y=stock_data_specific["Close"].astype("float"),
                             name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data_specific["Date"],
                             y=revenue_data_specific["Revenue"].astype("float"),
                             name="Revenue"), row=2, col=1)

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False,
                      height=900,
                      title=stock,
                      xaxis_rangeslider_visible=True)
    fig.show()

## Question 1: Use yfinance to Extract Stock Data

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2: Use Webscraping to Extract Tesla Revenue Data

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
html_data = requests.get(url).text

soup = BeautifulSoup(html_data, "html.parser")

rows = []
for table in soup.find_all("table"):
    if "Tesla Quarterly Revenue" in table.get_text():
        for row in table.find("tbody").find_all("tr"):
            cols = row.find_all("td")
            if len(cols) == 2:
                rows.append({"Date": cols[0].text.strip(), "Revenue": cols[1].text.strip()})
        break
tesla_revenue = pd.DataFrame(rows, columns=["Date", "Revenue"])

tesla_revenue["Revenue"] = tesla_revenue["Revenue"].str.replace(r",|\$", "", regex=True)
tesla_revenue.dropna(inplace=True)
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"] != ""]

tesla_revenue.tail()

## Question 3: Use yfinance to Extract Stock Data

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4: Use Webscraping to Extract GME Revenue Data

In [ ]:
url_2 = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
html_data_2 = requests.get(url_2).text

soup_2 = BeautifulSoup(html_data_2, "html.parser")

rows = []
for table in soup_2.find_all("table"):
    if "GameStop Quarterly Revenue" in table.get_text():
        for row in table.find("tbody").find_all("tr"):
            cols = row.find_all("td")
            if len(cols) == 2:
                rows.append({"Date": cols[0].text.strip(), "Revenue": cols[1].text.strip()})
        break
gme_revenue = pd.DataFrame(rows, columns=["Date", "Revenue"])

gme_revenue["Revenue"] = gme_revenue["Revenue"].str.replace(r",|\$", "", regex=True)
gme_revenue.dropna(inplace=True)
gme_revenue = gme_revenue[gme_revenue["Revenue"] != ""]

gme_revenue.tail()

## Question 5: Plot Tesla Stock Graph

In [ ]:
make_graph(tesla_data, tesla_revenue, 'Tesla')

## Question 6: Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, 'GameStop')